<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c40-header.png" alt="Nextia Learning · Experiments and Causal Impact: Did Our Change Work?" width="100%">

# Solution: Peeking

**[Peeking](https://learning.nextia-ai.com/courses/experiments/m04/peeking/)** · Experiments and Causal Impact: Did Our Change Work? · Module 4, lesson 1 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** measure, by simulation, how often "look every day and stop at the first p < 0.05" declares a winner when nothing changed, and how a sequential test lets you look every day without that problem.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | p-values and intervals, from [The result and its uncertainty](https://learning.nextia-ai.com/courses/experiments/m03/the-result-and-its-uncertainty/); A/A tests, from [Check the test before the result](https://learning.nextia-ai.com/courses/experiments/m03/check-the-test-before-the-result/). |
| **Tested** | Python 3.12 with pandas 3.1.1, SciPy 1.15.3, statsmodels 0.15.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m04/peeking/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C40/M04-L01-peeking/peeking-solution.ipynb).

## Setup: install the packages

Run the next cell first. It installs the versions that the course uses: pandas 3.1.1, SciPy 1.15.3 and statsmodels 0.15.0, and matplotlib 3.11.2 for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with these versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 scipy==1.15.3 statsmodels==0.15.0 matplotlib==3.11.2
import importlib.metadata as _md
for _name in ["pandas", "scipy", "statsmodels", "matplotlib"]:
    print(_name, _md.version(_name))

## Setup: the project folder

This notebook makes its own data with small simulations, so it downloads nothing. Run the next cell. It makes a folder `larkfield-experiments` and works in it.

You should see `Ready: 0 files in data/.`

In [ ]:
# Setup: download the course data into larkfield-experiments/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

DATA = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C40/data/"
FILES = {}
LOCAL = os.environ.get("NX_DATA_LOCAL")  # authors only: a local copy of the labs data
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("larkfield-experiments").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    if LOCAL:
        shutil.copy(Path(LOCAL) / name, path)
    else:
        download(DATA + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-experiments and run again.")
os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in data/.")

## Setup: helpers

The next cell loads pandas as `pd`, NumPy as `np`, SciPy's `stats` and statsmodels' formula interface as `smf`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `truth(name)` reads a scenario's `truth.json`: the simulator's true effects.

Run the cell. It prints nothing.

In [ ]:
# Helpers: checks that do not show the answer, and the simulator's truth.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def truth(name):
    """The simulator's true effects for a scenario (a real test never has this)."""
    return json.loads(open(f"data/{name}/truth.json").read())

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. One A/A test, looked at every day

An A/A test shows both groups the same shop: any difference is chance. The next cell simulates one A/A test of the order rate (22% of customers order), with 2,000 new customers per group each day for 28 days, and computes the usual z-test each day on everything seen so far.

> **Predict.** Out of 28 daily looks, how often will this test show p < 0.05?

In [ ]:
def aa_tests(n_tests, days=28, per_day=2000, rate=0.22, seed=0):
    """Cumulative order rates of A and B and the daily p-values of n_tests A/A tests."""
    rng = np.random.default_rng(seed)
    xa = rng.binomial(per_day, rate, (n_tests, days)).cumsum(axis=1)
    xb = rng.binomial(per_day, rate, (n_tests, days)).cumsum(axis=1)
    m = per_day * np.arange(1, days + 1)
    pool = (xa + xb) / (2 * m)
    var = pool * (1 - pool) * 2 / m
    diff = xb / m - xa / m
    p = 2 * stats.norm.sf(np.abs(diff) / np.sqrt(var))
    return diff, var, p

diff, var, p = aa_tests(1)
print(np.round(p[0], 3))
print("days with p < 0.05:", int((p[0] < 0.05).sum()))

> **Check.** You should see 28 p-values and `days with p < 0.05: 12`.

## 2. Many A/A tests: the false-winner rate

Now 2,000 A/A tests. "Fixed length" looks once, on day 28. "Peeking" stops at the first day with p < 0.05.

In [ ]:
diff, var, p = aa_tests(2000)
fixed = (p[:, -1] < 0.05).mean()
peek = (p < 0.05).any(axis=1).mean()
print(f"fixed length: {fixed:.1%} false winners; peeking every day: {peek:.1%} false winners")

> **Check.** You should see `fixed length: 4.5% false winners; peeking every day: 27.2% false winners`.

The 5% rule holds only if you look once. Each look is another chance for noise to cross the line. The course's run with 10,000 tests gives 27.6% for daily peeking.

## 3. A sequential test: look every day, keep the 5%

A **sequential test** widens the interval at each look by just enough that the chance of ever excluding the truth stays at 5%, over all the looks together. This is the mixture sequential probability ratio test (mSPRT): `tau` is a guess at the size of a real effect.

In [ ]:
def always_valid_p(diff, var, tau):
    """The mSPRT p-value at each look (running minimum): 1 / the mixture likelihood ratio."""
    lr = np.sqrt(var / (var + tau**2)) * np.exp(diff**2 * tau**2 / (2 * var * (var + tau**2)))
    return np.minimum.accumulate(np.minimum(1, 1 / lr), axis=1)

seq_p = always_valid_p(diff, var, tau=0.05 * 0.22)
print(f"sequential test, looking every day: {(seq_p < 0.05).any(axis=1).mean():.1%} false winners")

> **Check.** You should see `sequential test, looking every day: 1.5% false winners`, below 5%.

The cost: with a real effect, the sequential test usually needs more days to decide than a fixed test needs at its end. You buy the right to look every day.

## 4. Your turn: peek once a week

> **Your turn.** Tomás agrees to look only on days 7, 14, 21 and 28. Compute the share of the 2,000 A/A tests in `p` that show p < 0.05 on at least one of these days. Put it in `weekly_rate`, rounded to 3 decimals. (Day 7 is column 6.)

In [ ]:
weekly_rate = round(float((p[:, [6, 13, 20, 27]] < 0.05).any(axis=1).mean()), 3)
print(weekly_rate)

In [ ]:
expect('weekly_rate', weekly_rate, 0.12)

Fewer looks, fewer false winners, but still more than 5%. Only one look, or a sequential test, keeps the promise.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/experiments/m04/peeking/) explains always-valid intervals with a picture, and has the knowledge checks that count toward your certificate. In the project, `xp peek --aa-tests 1000` runs this simulation, and `xp peek early-stop` shows a real-looking test that was stopped on day 3.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Many metrics and many tests](https://learning.nextia-ai.com/courses/experiments/m04/many-metrics-and-many-tests/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/experiments/m04/peeking/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.